In [1]:
"""
03_canonical_convnext.py / 03_canonical_convnext.ipynb

Consolidated Benchmark: Geometric Projections vs. Standard Normalization in Canonical ConvNeXt
Dataset: CIFAR-10 (45,000 Train / 5,000 Validation / 10,000 Blind Test)
Protocol: 3 Independent Seeds [42, 1337, 2026], 12 Epochs with Data Augmentation and Cosine Annealing

Architectures Evaluated:
  - LayerNeXt (Baseline): Standard ConvNeXt with affine LayerNorm2d, standard Conv2d, GELU
  - EquatorialNeXt: Intra-sample zero-trace projection (S^{C-2}) + Weight Standardization + SiLU
  - ConicalNeXt: Radial spherical projection (S^{C-1}) + Cosine-normalized Conv2d + GELU

Diagnostics Audited:
  - Blind Test Accuracy and Cross-Entropy Loss
  - Latent Stable Rank (||H||_F^2 / ||H||_2^2)
  - Dead Channel Ratio (% channels with variance < 1e-4)
  - Gradient Signal-to-Noise Ratio (Grad SNR)
  - Loss Landscape Sharpness (Delta L under 2% relative parameter noise)
"""

import os
import gc
import time
import pickle
from typing import Dict, List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, TensorDataset
from torchvision import transforms

# -----------------------------------------------------------------------------
# 0. Global Setup and Hardware Configuration
# -----------------------------------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEEDS: List[int] = [42, 1337, 2026]
EPOCHS: int = 12
BATCH_SIZE: int = 128
LR: float = 1e-3
WEIGHT_DECAY: float = 1e-4
EPS: float = 1e-7

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True


# -----------------------------------------------------------------------------
# 1. Dataset Loader & Preprocessing Pipeline
# -----------------------------------------------------------------------------
def load_cifar10_batch(filepath: str) -> Tuple[np.ndarray, np.ndarray]:
    with open(filepath, "rb") as f:
        batch = pickle.load(f, encoding="bytes")
    return batch[b"data"], np.array(batch[b"labels"], dtype=np.int64)


def get_cifar10_splits(
    val_size: int = 5000,
    split_seed: int = 42
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """Loads and partitions CIFAR-10 into disjoint Train, Validation, and Test sets."""
    base_path = "/kaggle/input/datasets/pankrzysiu/cifar10-python/cifar-10-batches-py"
    if not os.path.exists(base_path):
        base_path = "/kaggle/input/cifar10-python/cifar-10-batches-py"

    if os.path.exists(base_path):
        x_all_list, y_all_list = [], []
        for i in range(1, 6):
            x, y = load_cifar10_batch(os.path.join(base_path, f"data_batch_{i}"))
            x_all_list.append(x)
            y_all_list.append(y)
        x_all = np.concatenate(x_all_list)
        y_all = np.concatenate(y_all_list)
        x_test, y_test = load_cifar10_batch(os.path.join(base_path, "test_batch"))
    else:
        from torchvision import datasets
        train_ds = datasets.CIFAR10(root="./data", train=True, download=True)
        test_ds = datasets.CIFAR10(root="./data", train=False, download=True)
        x_all = train_ds.data.transpose(0, 3, 1, 2).reshape(50000, 3072)
        y_all = np.array(train_ds.targets, dtype=np.int64)
        x_test = test_ds.data.transpose(0, 3, 1, 2).reshape(10000, 3072)
        y_test = np.array(test_ds.targets, dtype=np.int64)

    rng = np.random.RandomState(split_seed)
    indices = rng.permutation(len(x_all))
    val_indices, train_indices = indices[:val_size], indices[val_size:]

    x_train, y_train = x_all[train_indices], y_all[train_indices]
    x_val, y_val = x_all[val_indices], y_all[val_indices]

    train_mean = np.mean(x_train, axis=0, keepdims=True)
    train_std = np.std(x_train, axis=0, keepdims=True) + 1e-8

    x_train = ((x_train - train_mean) / train_std).reshape(-1, 3, 32, 32).astype(np.float32)
    x_val = ((x_val - train_mean) / train_std).reshape(-1, 3, 32, 32).astype(np.float32)
    x_test = ((x_test - train_mean) / train_std).reshape(-1, 3, 32, 32).astype(np.float32)

    return (
        torch.from_numpy(x_train), torch.from_numpy(y_train),
        torch.from_numpy(x_val), torch.from_numpy(y_val),
        torch.from_numpy(x_test), torch.from_numpy(y_test)
    )


class AugmentedCIFARDataset(Dataset):
    def __init__(self, data_tensor: torch.Tensor, targets_tensor: torch.Tensor, transform=None) -> None:
        self.data = data_tensor
        self.targets = targets_tensor
        self.transform = transform

    def __len__(self) -> int:
        return len(self.targets)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        x = self.data[idx]
        y = self.targets[idx]
        if self.transform:
            x = self.transform(x)
        return x, y


# -----------------------------------------------------------------------------
# 2. Normalization & Convolutional Primitives
# -----------------------------------------------------------------------------
class LayerNorm2d(nn.Module):
    """Channel-wise 2D LayerNorm with learnable affine parameters (official ConvNeXt)."""
    def __init__(self, dim: int, eps: float = 1e-6) -> None:
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.bias = nn.Parameter(torch.zeros(dim))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        u = x.mean(1, keepdim=True)
        s = (x - u).pow(2).mean(1, keepdim=True)
        x_norm = (x - u) / torch.sqrt(s + self.eps)
        return self.weight[:, None, None] * x_norm + self.bias[:, None, None]


class EquatorialNorm2d(nn.Module):
    """Zero-trace intra-sample projection on S^{C-2} without affine parameters."""
    def __init__(self, dim: int, eps: float = 1e-6) -> None:
        super().__init__()
        self.eps = eps
        self.scale = float(dim ** 0.5)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        u = x.mean(dim=1, keepdim=True)
        x_cent = x - u
        norm = torch.norm(x_cent, p=2, dim=1, keepdim=True) + self.eps
        return self.scale * (x_cent / norm)


class ConicalNorm2d(nn.Module):
    """Radial spherical projection on S^{C-1} without affine parameters."""
    def __init__(self, dim: int, eps: float = 1e-7) -> None:
        super().__init__()
        self.eps = eps
        self.scale = float(dim ** 0.5)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        norm = torch.norm(x, p=2, dim=1, keepdim=True) + self.eps
        return self.scale * (x / norm)


class WSConv2d(nn.Conv2d):
    """1x1 convolution with fused Weight Standardization."""
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        w = self.weight
        w_mean = w.mean(dim=(1, 2, 3), keepdim=True)
        w_cent = w - w_mean
        w_norm = w_cent / (torch.norm(w_cent, p=2, dim=(1, 2, 3), keepdim=True) + 1e-5)
        return F.conv2d(x, w_norm, self.bias, self.stride, self.padding, self.dilation, self.groups)


class CosineConv2d(nn.Conv2d):
    """1x1 convolution with L2 cosine weight normalization."""
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        w_norm = F.normalize(self.weight, p=2, dim=(1, 2, 3), eps=1e-7)
        return F.conv2d(x, w_norm, self.bias, self.stride, self.padding, self.dilation, self.groups)


# -----------------------------------------------------------------------------
# 3. Canonical ConvNeXt Architecture Factory
# -----------------------------------------------------------------------------
class CanonicalNeXtBlock(nn.Module):
    """
    Canonical ConvNeXt block adhering to Liu et al. (CVPR 2022):
      DW 7x7 -> Normalization -> PW 1x1 (4x) -> Activation -> PW 1x1 -> LayerScale
    """
    def __init__(self, dim: int, mode: str = "layernext", layer_scale_init: float = 1e-6) -> None:
        super().__init__()
        self.mode = mode
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim, bias=False)

        if mode == "layernext":
            self.norm = LayerNorm2d(dim)
            self.pw1 = nn.Conv2d(dim, 4 * dim, kernel_size=1, bias=False)
            self.act = nn.GELU()
            self.pw2 = nn.Conv2d(4 * dim, dim, kernel_size=1, bias=False)
        elif mode == "equatorialnext":
            self.norm = EquatorialNorm2d(dim)
            self.pw1 = WSConv2d(dim, 4 * dim, kernel_size=1, bias=False)
            self.act = nn.SiLU()
            self.pw2 = WSConv2d(4 * dim, dim, kernel_size=1, bias=False)
        elif mode == "conicalnext":
            self.norm = ConicalNorm2d(dim)
            self.pw1 = CosineConv2d(dim, 4 * dim, kernel_size=1, bias=False)
            self.act = nn.GELU()
            self.pw2 = CosineConv2d(4 * dim, dim, kernel_size=1, bias=False)
        else:
            raise ValueError(f"Unknown mode: {mode}")

        self.gamma = nn.Parameter(layer_scale_init * torch.ones((dim, 1, 1)), requires_grad=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        input_x = x
        h = self.dwconv(x)
        h = self.norm(h)
        h = self.pw1(h)
        h = self.act(h)
        h = self.pw2(h)
        return input_x + self.gamma * h


class CanonicalNeXtNet(nn.Module):
    """Full ConvNeXt network with 3 downsampling stages."""
    def __init__(
        self,
        mode: str = "layernext",
        num_classes: int = 10,
        dims: List[int] = [32, 64, 128],
        depths: List[int] = [2, 2, 2]
    ) -> None:
        super().__init__()
        self.mode = mode

        def create_norm(d: int) -> nn.Module:
            if mode == "layernext":
                return LayerNorm2d(d)
            elif mode == "equatorialnext":
                return EquatorialNorm2d(d)
            elif mode == "conicalnext":
                return ConicalNorm2d(d)
            raise ValueError(f"Unknown mode: {mode}")

        self.stem = nn.Sequential(
            nn.Conv2d(3, dims[0], kernel_size=3, padding=1),
            create_norm(dims[0])
        )

        self.stages = nn.ModuleList()
        self.downsamples = nn.ModuleList()

        for i in range(len(dims)):
            stage = nn.Sequential(*[
                CanonicalNeXtBlock(dim=dims[i], mode=mode) for _ in range(depths[i])
            ])
            self.stages.append(stage)

            if i < len(dims) - 1:
                downsample = nn.Sequential(
                    create_norm(dims[i]),
                    nn.Conv2d(dims[i], dims[i + 1], kernel_size=2, stride=2)
                )
                self.downsamples.append(downsample)

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.norm_final = create_norm(dims[-1])
        self.head = nn.Linear(dims[-1], num_classes)

    def extract_features(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        for i in range(len(self.stages)):
            x = self.stages[i](x)
            if i < len(self.downsamples):
                x = self.downsamples[i](x)
        x = self.norm_final(self.pool(x))
        return torch.flatten(x, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.extract_features(x))


# -----------------------------------------------------------------------------
# 4. Diagnostic & Health Audit Functions
# -----------------------------------------------------------------------------
def audit_health(
    model: nn.Module,
    criterion: nn.Module,
    test_loader: DataLoader,
    train_loader: DataLoader,
    device: torch.device
) -> Tuple[float, float, float, float]:
    model.eval()

    # 4.1 Latent Space Audit
    latents_list: List[torch.Tensor] = []
    with torch.no_grad():
        for xb, _ in test_loader:
            xb = xb.to(device)
            latents_list.append(model.extract_features(xb).cpu())
    latents = torch.cat(latents_list, dim=0)

    h_centered = latents - latents.mean(dim=0, keepdim=True)
    _, s, _ = torch.svd(h_centered)
    srank = float((torch.sum(s ** 2) / (torch.max(s) ** 2 + EPS)).item())
    dead_channels = float((torch.var(latents, dim=0) < 1e-4).float().mean().item() * 100.0)

    # 4.2 Loss Landscape Sharpness (Delta L under 2% relative parameter noise)
    base_loss = 0.0
    total_samples = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            base_loss += criterion(model(xb), yb).item() * yb.size(0)
            total_samples += yb.size(0)
    base_loss /= total_samples

    orig_weights = {n: p.data.clone() for n, p in model.named_parameters()}
    with torch.no_grad():
        for n, p in model.named_parameters():
            p.data.add_(torch.randn_like(p) * (0.02 * p.data.norm(2) / (p.numel() ** 0.5 + EPS)))

    perturbed_loss = 0.0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            perturbed_loss += criterion(model(xb), yb).item() * yb.size(0)

    with torch.no_grad():
        for n, p in model.named_parameters():
            p.data.copy_(orig_weights[n])

    sharpness = max(0.0, (perturbed_loss / total_samples) - base_loss)

    # 4.3 Gradient Signal-to-Noise Ratio (SNR)
    model.train()
    batch_grads: List[torch.Tensor] = []
    for idx, (xb, yb) in enumerate(train_loader):
        if idx >= 15:
            break
        xb, yb = xb.to(device), yb.to(device)
        model.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        flat = [p.grad.data.view(-1) for p in model.parameters() if p.grad is not None]
        batch_grads.append(torch.cat(flat))

    grads_tensor = torch.stack(batch_grads)
    mean_g = grads_tensor.mean(dim=0)
    std_g = grads_tensor.std(dim=0) + EPS
    grad_snr = float((mean_g.norm(2) / (std_g.norm(2) + EPS)).item())

    return srank, dead_channels, grad_snr, float(sharpness)


# -----------------------------------------------------------------------------
# 5. Multi-Seed Benchmark Execution Loop
# -----------------------------------------------------------------------------
def run_benchmark() -> None:
    x_tr, y_tr, x_va, y_va, x_te, y_te = get_cifar10_splits()

    train_transform = transforms.Compose([
        transforms.RandomCrop(32, padding=4, padding_mode="reflect"),
        transforms.RandomHorizontalFlip(),
    ])

    val_loader = DataLoader(TensorDataset(x_va, y_va), batch_size=256, shuffle=False)
    test_loader = DataLoader(TensorDataset(x_te, y_te), batch_size=256, shuffle=False)

    configs = [
        ("LayerNeXt (Baseline)", "layernext"),
        ("EquatorialNeXt", "equatorialnext"),
        ("ConicalNeXt", "conicalnext"),
    ]

    results = {
        cfg[0]: {
            "acc": [], "loss": [], "srank": [],
            "dead": [], "snr": [], "sharp": [], "time": []
        }
        for cfg in configs
    }

    print("=" * 125)
    print(f"[INFO] Multi-Seed ConvNeXt Benchmark | Device: {DEVICE} | Seeds: {len(SEEDS)} | Epochs: {EPOCHS}")
    print("=" * 125)

    for seed_idx, seed in enumerate(SEEDS, 1):
        print(f"\n[INFO] Seed [{seed_idx}/{len(SEEDS)}]: {seed}")
        print("-" * 125)

        g = torch.Generator().manual_seed(seed)
        train_loader = DataLoader(
            AugmentedCIFARDataset(x_tr, y_tr, transform=train_transform),
            batch_size=BATCH_SIZE, shuffle=True, drop_last=True, generator=g
        )

        for name, mode in configs:
            torch.manual_seed(seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(seed)

            model = CanonicalNeXtNet(mode=mode).to(DEVICE)
            optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
            criterion = nn.CrossEntropyLoss()

            t0 = time.time()
            for _ in range(1, EPOCHS + 1):
                model.train()
                for xb, yb in train_loader:
                    xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                    optimizer.zero_grad()
                    loss = criterion(model(xb), yb)
                    loss.backward()
                    optimizer.step()
                scheduler.step()

            elapsed = time.time() - t0

            # Blind Test Evaluation (10,000 samples)
            model.eval()
            test_correct, test_total, test_loss_sum = 0, 0, 0.0
            with torch.no_grad():
                for xb, yb in test_loader:
                    xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                    preds = model(xb)
                    test_loss_sum += criterion(preds, yb).item() * yb.size(0)
                    test_correct += (preds.argmax(dim=-1) == yb).sum().item()
                    test_total += yb.size(0)

            test_acc = (test_correct / test_total) * 100.0
            test_loss = test_loss_sum / test_total

            srank, dead, snr, sharp = audit_health(model, criterion, test_loader, train_loader, DEVICE)

            results[name]["acc"].append(test_acc)
            results[name]["loss"].append(test_loss)
            results[name]["srank"].append(srank)
            results[name]["dead"].append(dead)
            results[name]["snr"].append(snr)
            results[name]["sharp"].append(sharp)
            results[name]["time"].append(elapsed)

            print(
                f"  [RUN] {name:<22} -> Acc: {test_acc:>6.2f}% | Loss: {test_loss:.4f} | "
                f"Rank: {srank:>5.2f}/128 | SNR: {snr:.3f} | Delta-L: {sharp:.4f} ({elapsed:>5.1f}s)"
            )

            del model, optimizer, scheduler, criterion
            torch.cuda.empty_cache()

    # -------------------------------------------------------------------------
    # 6. Tabulated Performance Reports
    # -------------------------------------------------------------------------
    print("\n" + "=" * 125)
    print(f"EMPIRICAL PERFORMANCE SUMMARY ({len(SEEDS)} SEEDS, MEAN +/- STD)")
    print("=" * 125)
    print(f"{'ARCHITECTURE':<24} | {'TEST ACC (%)':<16} | {'TEST LOSS':<15} | {'STABLE RANK':<14} | {'GRAD SNR':<12} | {'TIME/SEED'}")
    print("-" * 125)
    for name, _ in configs:
        acc_m, acc_s = np.mean(results[name]["acc"]), np.std(results[name]["acc"])
        loss_m, loss_s = np.mean(results[name]["loss"]), np.std(results[name]["loss"])
        sr_m, sr_s = np.mean(results[name]["srank"]), np.std(results[name]["srank"])
        snr_m, snr_s = np.mean(results[name]["snr"]), np.std(results[name]["snr"])
        t_m = np.mean(results[name]["time"])

        print(
            f"{name:<24} | {acc_m:>6.2f}% +/- {acc_s:<5.2f} | {loss_m:>6.4f} +/- {loss_s:<5.4f} | "
            f"{sr_m:>5.2f} +/- {sr_s:<4.2f} | {snr_m:>5.3f} +/- {snr_s:<4.3f} | {t_m:>5.1f}s"
        )
    print("=" * 125)

    print("\n" + "=" * 85)
    print("COMPLEMENTARY HEALTH AUDIT: LANDSCAPE SHARPNESS & DEAD CHANNELS")
    print("=" * 85)
    print(f"{'ARCHITECTURE':<24} | {'SHARPNESS DELTA-L':<20} | {'DEAD CHANNELS (%)'}")
    print("-" * 85)
    for name, _ in configs:
        sh_m, sh_s = np.mean(results[name]["sharp"]), np.std(results[name]["sharp"])
        dead_m = np.mean(results[name]["dead"])
        print(f"{name:<24} | {sh_m:>7.4f} +/- {sh_s:<9.4f} | {dead_m:>12.2f}%")
    print("=" * 85)


if __name__ == "__main__":
    run_benchmark()

[INFO] Multi-Seed ConvNeXt Benchmark | Device: cuda | Seeds: 3 | Epochs: 12

[INFO] Seed [1/3]: 42
-----------------------------------------------------------------------------------------------------------------------------
  [RUN] LayerNeXt (Baseline)   -> Acc:  76.34% | Loss: 0.6838 | Rank:  4.72/128 | SNR: 0.281 | Delta-L: 0.0049 (281.8s)
  [RUN] EquatorialNeXt         -> Acc:  76.98% | Loss: 0.6637 | Rank:  5.15/128 | SNR: 0.247 | Delta-L: 0.0029 (257.5s)
  [RUN] ConicalNeXt            -> Acc:  76.29% | Loss: 0.6799 | Rank:  4.68/128 | SNR: 0.250 | Delta-L: 0.0043 (239.8s)

[INFO] Seed [2/3]: 1337
-----------------------------------------------------------------------------------------------------------------------------
  [RUN] LayerNeXt (Baseline)   -> Acc:  76.57% | Loss: 0.6721 | Rank:  4.55/128 | SNR: 0.279 | Delta-L: 0.0043 (283.8s)
  [RUN] EquatorialNeXt         -> Acc:  76.93% | Loss: 0.6605 | Rank:  4.88/128 | SNR: 0.264 | Delta-L: 0.0044 (257.2s)
  [RUN] ConicalNeXt     